# Convert raw to dsToolbox format

In [ ]:
from pathlib import Path
import sys

import pandas as pd

import cellpy
from cellpy.utils import plotutils

cellpy.__version__

In [ ]:
cur_dir = Path(".")
raw_data_path = cur_dir / "combined_protocol_results_realistic.bdf.csv"
assert raw_data_path.exists()

In [ ]:
c = cellpy.get(
    raw_data_path,
    instrument="batmo_bdf",
    cycle_mode="full_cell",
    mass=1.0,
    nominal_capacity=120.0,
    nom_cap_specifics="absolute",
)

In [ ]:
c.data.raw.head()

In [ ]:
c.add_to_summary("Maximum Cell Temperature / degC");

In [ ]:
plotutils.summary_plot(c)

In [ ]:
plotutils.summary_plot(c, y="capacities_absolute_with_rate", interactive=True, filters={"rate": (0.49, 0.51)}, hover_columns=["Maximum Cell Temperature / degC"])

In [ ]:
from functools import partial

def tweak_something(df: pd.DataFrame, steps=None) -> pd.DataFrame:
    if steps is not None:
        df = df.query("step_index in @steps")
    df["step_index"] = (
    df.groupby("cycle_index")["step_index"]
      .transform(lambda s: pd.factorize(s)[0] + 1)
      .astype("int64")
)
    df.current = -df.current
    return df

In [ ]:
plotutils.cycle_info_plot(c, cycle=2)

In [ ]:
c.to_bdf(cur_dir / "BattMo_OCV_P15_S3.csv", cycles=[2], extras=True, preprocess_fn=partial(tweak_something, steps=[6, 7, 8]))

In [ ]:
c.to_bdf(cur_dir / "BattMo_OCV_P15_S1.csv", cycles=[2], extras=True, preprocess_fn=partial(tweak_something, steps=[8, 9, 10]))